# Phase 1 — ANALYSIS

Reads only saved result files. Safe to re-run repeatedly; never re-runs an experiment.

Produces the numbers needed to slot Random Search into Tables III-V of the paper,
each against **its own** chance baseline (decision 4: baselines are per-method,
because expected overlap depends on subset size).


## 1. Config and imports

In [ ]:
from pathlib import Path
import sys, json, numpy as np

PROJ_ROOT = Path.cwd()
if not (PROJ_ROOT/"src").exists() and (PROJ_ROOT.parent/"src").exists():
    PROJ_ROOT = PROJ_ROOT.parent
SRC_DIR, RESULTS_DIR = PROJ_ROOT/"src", PROJ_ROOT/"results"
sys.path.insert(0, str(SRC_DIR))

import stability as stab
from stability_ci import nogueira_stability, pairwise_jaccard

rs      = json.load(open(RESULTS_DIR/"phase1_random_search.json"))
budget  = json.load(open(RESULTS_DIR/"phase1_measured_budget.json"))
nsga    = json.load(open(RESULTS_DIR/"final_raw_results.json"))
bc_nsga = json.load(open(RESULTS_DIR/"breast_cancer_matched_result.json"))
matched = json.load(open(RESULTS_DIR/"matched_seed_control.json"))
DATASETS = ["breast_cancer","colon","leukemia"]
P = {d: budget[d]["p"] for d in DATASETS}
print("loaded. datasets:", DATASETS)

## 2. Shared helpers — same reductions as the paper

In [ ]:
def front_tuples(front_dicts, p):
    return [(np.isin(np.arange(p), s["features"]), s["acc"], s["n_sel"])
            for s in front_dicts if s["n_sel"] > 0]

def masks_from_runs(runs, p, level):
    """level='front' -> union over the Pareto front; 'point' -> knee subset."""
    Z=[]
    for r in runs:
        fd = r["front"] if isinstance(r, dict) and "front" in r else r
        tup = front_tuples(fd, p)
        if not tup: continue
        if level == "front":
            u = np.zeros(p, bool)
            for m,_,_ in tup: u |= m
        else:
            u = stab.knee_point_mask(tup)
            if u is None: continue
            u = np.asarray(u, bool)
        Z.append(u)
    return np.array(Z)

def chance_stats(Z, p, n_draw=20000, seed=0):
    """Chance baseline computed from THIS method's own subset sizes."""
    rng = np.random.default_rng(seed); ks = Z.sum(1); jac=[]; zero=0
    for _ in range(n_draw):
        i,j = rng.choice(len(Z), 2, replace=False)
        a = rng.choice(p, ks[i], replace=False); b = rng.choice(p, ks[j], replace=False)
        it = len(np.intersect1d(a,b)); jac.append(it/(ks[i]+ks[j]-it)); zero += (it==0)
    return float(np.mean(jac)), 100*zero/n_draw

def observed_stats(Z):
    jac=[]; zero=0; tot=0
    for i in range(len(Z)):
        for j in range(i+1, len(Z)):
            it = np.logical_and(Z[i],Z[j]).sum(); un = np.logical_or(Z[i],Z[j]).sum()
            jac.append(it/un if un else 0.0); tot+=1; zero += (it==0)
    return float(np.median(jac)), float(np.mean(jac)), 100*zero/tot

## 3. Table — Random Search vs NSGA-II, each against its own chance baseline

In [ ]:
rows=[]
for ds in DATASETS:
    p = P[ds]
    nsga_boot = (bc_nsga if ds=="breast_cancer" else nsga[ds])["bootstrap_fixed"]["raw_runs"]
    sources = {
        ("NSGA-II","data resampling"): nsga_boot,
        ("NSGA-II","fixed draw")     : matched[ds]["raw_runs"],
        ("Random Search","data resampling"): rs[f"{ds}|bootstrap"]["runs"],
        ("Random Search","fixed draw")     : rs[f"{ds}|fixed"]["runs"],
    }
    for (meth,cond), runs in sources.items():
        for level in ["front","point"]:
            Z = masks_from_runs(runs, p, level)
            if len(Z) < 2: continue
            phi,_ = nogueira_stability(Z)
            jm, jmean, zero = observed_stats(Z)
            cj, cz = chance_stats(Z, p)
            rows.append(dict(dataset=ds, method=meth, condition=cond, level=level,
                             phi=round(phi,4), jac_med=round(jm,4), jac_mean=round(jmean,4),
                             chance_jac=round(cj,4), zero_pct=round(zero,0),
                             chance_zero_pct=round(cz,0), mean_k=round(float(Z.sum(1).mean()),1)))

hdr=["dataset","method","condition","level","phi","jac_med","jac_mean","chance_jac","zero_pct","chance_zero_pct","mean_k"]
print(" | ".join(f"{h:>15}" for h in hdr))
for r in rows: print(" | ".join(f"{str(r[h]):>15}" for h in hdr))
json.dump(rows, open(RESULTS_DIR/"phase1_stability_table.json","w"), indent=2)

## 4. Held-out accuracy at matched cardinality (decision 6)

In [ ]:
import sys; sys.path.insert(0,str(SRC_DIR))
import data as dm, bootstrap as bs
sys.path.insert(0,str(RESULTS_DIR))

acc_rows=[]
for ds in DATASETS:
    p=P[ds]
    # Random Search OOB is already stored per run
    rs_boot=[r["oob_accuracy"] for r in rs[f"{ds}|bootstrap"]["runs"] if r["oob_accuracy"] is not None]
    rs_k   =[r["knee_n_features"] for r in rs[f"{ds}|bootstrap"]["runs"]]
    acc_rows.append(dict(dataset=ds, method="Random Search",
        oob_med=round(float(np.median(rs_boot)),4),
        oob_iqr=[round(float(np.percentile(rs_boot,25)),3), round(float(np.percentile(rs_boot,75)),3)],
        median_k=float(np.median(rs_k))))

# NSGA-II values are already published; restated here for side-by-side reading
published={"breast_cancer":(0.930,2.0),"colon":(0.680,4.0),"leukemia":(0.774,3.5)}
for ds in DATASETS:
    acc_rows.append(dict(dataset=ds, method="NSGA-II (published)",
                         oob_med=published[ds][0], oob_iqr=None, median_k=published[ds][1]))

for r in sorted(acc_rows, key=lambda x:(x["dataset"],x["method"])):
    print(f'{r["dataset"]:>14} {r["method"]:>22}  OOB {r["oob_med"]:.3f}  median k {r["median_k"]}')
json.dump(acc_rows, open(RESULTS_DIR/"phase1_accuracy_table.json","w"), indent=2)
print()
print("NOTE: cardinalities differ between methods; compare OOB accuracy only at")
print("comparable median k, and report the k values alongside every accuracy.")

## 5. Reading guide — what each outcome means

Pre-registered in the roadmap, restated so the interpretation is not chosen after the fact:

| Outcome | Meaning | Action |
|---|---|---|
| NSGA-II clearly better than RS (OOB accuracy or Phi) | search is doing real work; the budget objection is answered | proceed to Phase 2 (mRMR); no population sweep needed |
| Difference negligible | at this budget NSGA-II offers little over random sampling | report honestly; the stability findings stand but become budget-conditional; Phase 3 becomes necessary |
| RS better than NSGA-II | important and publishable, but changes the thesis | stop and revisit framing before continuing |

All differences are descriptive. No significance test is run or claimed.
